<!--
ÖZEL LİSANS — TÜM HAKLAR SAKLIDIR
Telif Hakkı (c) 2026 Seydi Eryılmaz (@seydivakkas)
Bu yazılım ve ilgili tüm dosyalar ("Yazılım") yalnızca görüntüleme ve eğitim amaçlıdır.
Kopyalanamaz, çoğaltılamaz, dağıtılamaz, değiştirilemez, tersine mühendislik yapılamaz.
-->

# GÜN 062: Templates Temel Mantığı (Parametrik Polimorfizm, Derleme Zamanı Kod Üretimi, Sıfır Ek Maliyet, Two-Phase Lookup ve Binary Bloat Yönetimi)

[![License: All Rights Reserved](https://img.shields.io/badge/license-All%20Rights%20Reserved-red?style=flat-square)](https://github.com/seydivakkas)
[![C++ Standard](https://img.shields.io/badge/C%2B%2B-20-blue?style=flat-square&logo=c%2B%2B)](https://en.cppreference.com/w/cpp/20)
[![Domain](https://img.shields.io/badge/Domain-Radar%20DSP%20%26%20Zero--Allocation%20RingBuffers-darkgreen?style=flat-square)]()
[![Safety](https://img.shields.io/badge/Safety-MISRA%20C%2B%2B%3A2023%20%7C%20DO--178C%20DAL--A-orange?style=flat-square)]()

---

## 🏛️ Akademik ve Kurumsal Bildirim
Bu modül, **ASELSAN, HAVELSAN, ROKETSAN, TUSAŞ ve STM** gibi savunma sanayii devlerinin görev-kritik radar dijital sinyal işleme (DSP), füze güdüm filtreleme ve aviyonik veri tamponlama mimarilerinde sıfır çalışma zamanı ek maliyeti (*Zero-Cost Abstraction*) ile jenerik ve yüksek performanslı kod üretimi sağlamak üzere tasarlanmıştır.

- **Müfredat Fazı:** Faz 1 (F1) — Modern C++20 Çekirdek Sistem Programlama
- **Ön Koşul Konu:** Gün 061 (`enum class` ile güçlü tipleme ve durum makineleri)
- **Pedagojik Derinlik:** 40-Nokta Tam Kapsam Doğrulaması (First Principles -> Derleyici Örnekleme Mimarisi -> Sabotaj -> Benchmarking -> Sektörel Standartlar)


## 1. 🎯 Bugünkü Problem: Kod Tekrarı, Makro Yan Etkileri ve void* Tip Körlüğü

Sistem programlamada bir algoritmanın (örneğin bir Kalman filtresinin, dairesel tamponun veya matematiksel `clamp` fonksiyonunun) birden fazla veri tipiyle (`float`, `double`, `int32_t`, `fixed_point`) çalışması gerektiğinde C ve eski dillerde üç ölümcül yöntem kullanılırdı:

### 1. Kaba Kuvvet Kod Tekrarı (Copy-Paste Duplication)
```c
float  clamp_float(float val, float low, float high);
double clamp_double(double val, double low, double high);
int    clamp_int(int val, int low, int high);
```
- **Felaket:** Aynı algoritma 10 farklı tip için 10 kez kopyalanır. Bir hata düzeltildiğinde 9 kopya unutulur. DRY ilkesi tamamen çiğnenir.

### 2. C Ön İşlemci Makroları (`#define`)
```c
#define CLAMP(v, lo, hi) (((v) < (lo)) ? (lo) : (((v) > (hi)) ? (hi) : (v)))
```
- **Felaket:** Makroların tip denetimi ve kapsamı (`scope`) yoktur. `CLAMP(i++, min, max)` yazıldığında `i` değişkeni birden fazla kez artırılır (**Double Evaluation Yan Etkisi**)! Füze önleme hesaplamasında hedefin koordinatı yanlış hesaplanır.

### 3. C-Style `void*` ve Tip Körlüğü (`qsort` Yöntemi)
```c
void process_buffer(void* data, size_t elem_size, size_t count);
```
- **Felaket:** Tip bilgisi derleyici seviyesinde tamamen yok edilir. Derleyici optimizasyon yapamaz, `inline` edemez. Her işlem için çalışma zamanında işaretçi yönlendirmesi (*pointer dereference*) yapılır. Yanlış tip cast edildiğinde program çöker.

---

### Bugünkü Çözüm: C++ Templates (Şablonlar)
C++ şablonları, **Parametrik Polimorfizm (Statik / Derleme Zamanı Polimorfizmi)** sağlar:
- Şablon bir fonksiyon veya sınıf **kodun kendisi değildir**; derleyici için bir **kod üretim kalıbıdır (blueprint)**.
- Derleyici, şablonu çağrıldığı her somut tür için derleme zamanında ayrı ayrı üretir (**Instantiation**).
- **Sıfır Çalışma Zamanı Maliyeti:** Üretilen kod, o tipe özel elle yazılmış en hızlı assembly kodu kadar hızlıdır; derleyici fonksiyonu tamamen satır içine alabilir (`inline`).


## 2. ❓ Bu Konuyu Neden Öğreniyoruz? (Savunma Sanayii Gerekçesi)

Savunma ve radar mühendisliğinde şablonlar, performans ve güvenilirliğin kalbidir:

### 1. Deterministik DSP Filtreleri ve Donanım Tipleri
ASELSAN radar işlemcilerinde aynı Darbe Sıkıştırma (*Pulse Compression*) veya Hızlı Fourier Dönüşümü (*FFT*) algoritması:
- PC simülasyonunda yüksek hassasiyet için `double`,
- DSP kartında yüksek hız için `float`,
- FPGA ve ASIC arayüzlerinde `FixedPoint<16, 8>` (sabit noktalı sayı) tipiyle derlenir.
Algoritma tek bir şablon (`template<typename SampleType>`) olarak yazılır; derleyici her donanım için sıfır maliyetle optimize edilmiş makine kodu basar.

### 2. Dinamik Belleksiz Sabit Kapasiteli Tamponlar (NTTP)
Uçuş emniyeti standartlarında (DO-178C DAL-A) dinamik bellek (`malloc` / `std::vector`) yasaktır.
`template<typename T, size_t Capacity>` kullanılarak, boyutu ve türü derleme zamanında sabitlenen `RingBuffer<RadarTrack, 256>` üretilir. Bellek tamamen stack'te tahsis edilir ve sıfır parçalanma garantilenir.


## 3. 🔍 5N1K Analizi (Sistem ve Donanım Sınırları)

| Soru | Sistem & Donanım Mühendisliği Yanıtı |
| :--- | :--- |
| **Ne? (What)** | **C++ Templates (Şablonlar)**, türleri ve derleme zamanı sabitlerini parametre olarak alabilen, derleyicinin ihtiyaç duyulan her somut tür için optimize edilmiş bağımsız makine kodu üretmesini sağlayan parametrik metaprogramlama aracıdır. |
| **Neden? (Why)** | Kod tekrarını ortadan kaldırmak, makroların ölümcül yan etkilerini ve `void*` tip körlüğünü yok etmek; çalışma zamanı sanal tablo (`vtable`) maliyeti ödemeden tip güvenli statik polimorfizm sağlamak için. |
| **Nasıl? (How)** | `template<typename T>` veya `template<typename T, size_t N>` sözdizimi ile tanımlanır. Kullanım anında derleyici somut türleri çıkarır (*Type Deduction*) ve ilgili kod varyantını derler (*Instantiation*). |
| **Nerede? (Where)** | Radar dijital sinyal işleme boru hatlarında, Aviyonik RingBuffer tamponlarında, Kalman filtrelerinde, STL (`std::vector`, `std::array`) ve Eigen lineer cebir kütüphanelerinde. |
| **Ne Zaman? (When)** | Bir algoritma veya veri yapısı mantıksal olarak üzerinde çalıştığı veri tipinden bağımsız olduğunda ve çalışma zamanı başarım kaybı istenmediğinde. |
| **Kim? (Who)** | DSP mühendisleri, aviyonik kütüphane mimarları, gömülü sistem programcıları ve sıfır maliyetli soyutlama uzmanları. |


## 4. 💡 Günlük Hayat Benzetmesi (Zihinsel Model / Analoji)

> ### ⚙️ Endüstriyel Parça Döküm Kalıbı ve CNC Şablonu Analojisi
> 
> Bir savunma sanayii mühimmat fabrikasını düşünün:
> 
> 1. **Eski C Kopyalama Yolu (Ayrı Ayrı Atölyeler):**
>    Fabrika çelik dişli üretmek için ayrı bir atölye, pirinç dişli üretmek için ayrı bir atölye, titanyum dişli için üçüncü bir atölye kurar. Atölyeler tamamen aynı tezgah düzenine sahiptir. Çelik atölyesinde yapılan bir iyileştirme titanyum atölyesine aktarılmazsa kalite bozulur (**Kod Tekrarı ve Bakım Krizi**).
> 
> 2. **C-Makro Yolu (Kağıt Üzerine Çizilmiş Ham Taslak):**
>    Usta bir kağıda dişli resmi çizer ve çırakların önüne atar. Hiçbir ölçü veya metalürji kuralı yoktur. Çırak pirinci fazla ısıtır, patlama olur veya ölçüler kayar (**Tip Denetimsiz Makro Yan Etkisi**).
> 
> 3. **C++ Template Yolu (Yüksek Hassasiyetli Çelik Döküm Kalıbı / Blueprint):**
>    Mühendisler fabrikanın merkezine tek bir mükemmel dişli kalıbı (`template<typename Metal>`) yerleştirir.
>    - Sipariş Çelik Dişli geldiğinde: Kalıba erimiş çelik dökülür; fabrikadan kusursuz bir çelik dişli çıkar (`Gear<Steel>`).
>    - Sipariş Titanyum Dişli geldiğinde: Aynı kalıba titanyum dökülür; fabrikadan titanyum dişli çıkar (`Gear<Titanium>`).
>    - Kalıbın kendisi dişli değildir; dişli üreten araçtır!
>    - Çıkan her dişli o metalin özelliklerine göre son derece sağlam ve pürüzsüzdür. Hiçbir aracı mekanizma veya ek dişli ağırlığı yoktur (**Sıfır Ek Maliyetli Soyutlama**)!


## 5. 🧱 İlk İlkeler (First Principles) ve Matematiksel / Donanımsal Model

### 5.1 Derleyicinin İki Aşamalı İsim Çözümleme (Two-Phase Name Lookup) Modeli

Bir şablon derlenirken derleyici kodu iki farklı zamanda inceler:

```text
[ 1. AŞAMA: Şablon Tanım Zamanı (Definition Time) ]
- Derleyici şablonun genel sözdizimini (syntax) denetler.
- Bağımsız İsimler (Non-dependent Names): Tür parametresi 'T'ye bağlı olmayan isimler derhal çözümlenir ve bağlanır.
- Henüz somut hiçbir makine kodu üretilmez!

[ 2. AŞAMA: Örnekleme Zamanı (Instantiation Time) ]
- Kullanıcı 'clamp<float>(...)' çağrısı yaptığında başlar.
- Bağımlı İsimler (Dependent Names): Tür parametresi 'T'ye bağlı tüm işlemler (örneğin T::value_type, v < min) somut tür için kontrol edilir.
- Türün ilgili operatörü (örneğin '<') destekleyip desteklemediği denetlenir. Desteklemiyorsa derleme durdurulur!
- Tür için optimize edilmiş somut assembly fonksiyonu oluşturulur.
```

---

### 5.2 Kod Şişmesi (Binary / Code Bloat) Matematiği

Şablonların çalışma zamanı maliyeti sıfırdır; ancak derleme zamanı ve ikili dosya (*binary size*) maliyeti vardır.
Bir şablon fonksiyonu $K$ bayt makine koduna dönüşüyorsa ve sistemde $N$ farklı tür için örneklenirse:

$$	ext{Total Code Size} = \sum_{i=1}^{N} 	ext{Size}(	ext{TemplateInstance}_i) pprox N 	imes K 	ext{ Bayt}$$

- Eğer bir gömülü sistemde 50 farklı tür için 2 KB'lık karmaşık bir şablon örneklendirilirse, Flash bellekte **100 KB** sadece bu şablonun kopyaları yer kaplar.
- **Donanımsal Etki:** Fazla kod şişmesi, işlemcinin komut önbelleğini (L1 Instruction Cache) kirleterek *Instruction Cache Miss* oranını artırabilir.
- **Çözüm (Template Hoisting):** Tipe bağlı olmayan ortak mantık (örneğin bellek kopyalama veya sınır kontrolleri) şablon dışı bir taban fonksiyona taşınır.


## 6. 📐 ASCII ve Mermaid Sistem Mimarisi Şeması

### 6.1 Derleyici Örnekleme Akışı (Compilation Pipeline)

```text
[ Kaynak Kod: template<typename T> T min(T a, T b) ]
                        │
                        ▼ (C++ Derleyici Ön Uç - Frontend)
         ┌──────────────┴──────────────┐
         ▼                             ▼
Çağrı: min(10, 20)             Çağrı: min(3.14, 2.71)
(T = int)                      (T = double)
         │                             │
         ▼                             ▼
Örnekleme 1:                   Örnekleme 2:
int min_int(int a, int b) {    double min_dbl(double a, double b) {
  return a < b ? a : b;          return a < b ? a : b;
}                              }
         │                             │
         ▼                             ▼
[ x86_64: cmovl komutu ]       [ SSE/AVX: minsd xmm0, xmm1 ]
```

---

### 6.2 Donanıma Özel Kod Üretimi ve Inlining (Mermaid)

```mermaid
flowchart TD
    A["Jenerik Radar Filtre Şablonu<br/>template&lt;typename T, size_t N&gt;"] --> B{Derleyici Örnekleme Motoru}

    B -->|T = float, N = 128| C["RadarFilter&lt;float, 128&gt;"]
    B -->|T = double, N = 512| D["RadarFilter&lt;double, 512&gt;"]

    C --> E[AVX-512 Vektorize Assembly]
    D --> F[FMA Yüksek Hassasiyet Assembly]

    E --> G[DSP Gerçek Zamanlı Yürütme: 1.2 us]
    F --> H[Aviyonik Yüksek Doğruluk: 4.8 us]

    style A fill:#1E293B,stroke:#38BDF8,stroke-width:2px,color:#fff
    style B fill:#065F46,stroke:#10B981,stroke-width:2px,color:#fff
    style C fill:#1E3A8A,stroke:#3B82F6,stroke-width:1.5px,color:#fff
    style D fill:#1E3A8A,stroke:#3B82F6,stroke-width:1.5px,color:#fff
```


In [ ]:
"""
ÖZEL LİSANS — TÜM HAKLAR SAKLIDIR
Telif Hakkı (c) 2026 Seydi Eryılmaz (@seydivakkas)
Bu yazılım ve ilgili tüm dosyalar ("Yazılım") yalnızca görüntüleme ve eğitim amaçlıdır.
Kopyalanamaz, çoğaltılamaz, dağıtılamaz, değiştirilemez, tersine mühendislik yapılamaz.
"""

# ==============================================================================
# MODÜL 1: Temel Çekirdek Mekanizma (Simulated Template Instantiation Engine)
# Derleyicinin Şablon Örnekleme ve Tip Çözümleme Mantığı Simülasyonu
# ==============================================================================

from typing import Dict, Any, Type

class CompilerTemplateEngine:
    """
    C++ Derleyicisinin Şablon (Template) Örnekleme Davranışını Simüle Eder.
    Şablon fonksiyon ilk çağrıldığında o tipe özel kod türetilir ve önbelleğe alınır.
    """
    def __init__(self, template_name: str, code_generator_func):
        self.name = template_name
        self.generator = code_generator_func
        self.instantiations: Dict[Type, Any] = {}
        self.instantiation_count = 0

    def __call__(self, arg1: Any, arg2: Any) -> Any:
        type1 = type(arg1)
        type2 = type(arg2)

        # Tip Çıkarsama (Type Deduction) & Uyuşmazlık Denetimi
        if type1 != type2:
            raise TypeError(
                f"[Derleyici Hatası - No Matching Function]: '{self.name}({type1.__name__}, {type2.__name__})' "
                f"şablon parametresi 'T' için tutarlı bir tür çıkarılamadı!"
            )

        # Örnekleme (Instantiation): Bu tür için daha önce kod üretildi mi?
        if type1 not in self.instantiations:
            # 2. Aşama (Phase 2): İlgili türe özel fonksiyonu derle
            specialized_func = self.generator(type1)
            self.instantiations[type1] = specialized_func
            self.instantiation_count += 1
            print(f"  [DERLEYİCİ]: '{self.name}<{type1.__name__}>' somut örneği derlendi! (Toplam: {self.instantiation_count})")

        return self.instantiations[type1](arg1, arg2)

def run_module_01_core():
    print("[*] Modül 1: Derleyici Şablon Örnekleme (Instantiation) Motoru Test Ediliyor...")

    # Jenerik Matematiksel Min Şablonu: template<typename T> T min(T a, T b)
    def min_code_generator(concrete_type: Type):
        return lambda a, b: a if a < b else b

    template_min = CompilerTemplateEngine("min", min_code_generator)

    # 1. Çağrı (int): Yeni örnekleme üretilir
    r1 = template_min(10, 20)
    assert r1 == 10
    assert template_min.instantiation_count == 1

    # 2. Çağrı (int): Önceden üretilen kod kullanılır (Örnekleme artmaz!)
    r2 = template_min(100, 50)
    assert r2 == 50
    assert template_min.instantiation_count == 1

    # 3. Çağrı (float): Yeni bir tür, yeni örnekleme üretilir!
    r3 = template_min(3.14, 2.71)
    assert r3 == 2.71
    assert template_min.instantiation_count == 2

    # 4. Hatalı Çağrı (int, float uyuşmazlığı): Derleme hatası fırlatır
    caught_deduction_error = False
    try:
        _ = template_min(42, 3.14)
    except TypeError as ex:
        caught_deduction_error = True
        print(f"  -> [BEKLENEN DERLEYİCİ HATASI]: {ex}")

    assert caught_deduction_error is True
    print("  -> [PASS] Modül 1 şablon örnekleme ve tip koruma mekanizması başarıyla doğrulandı.")

if __name__ == "__main__":
    run_module_01_core()


## 8. Modül 2: Donanıma Yakın / İleri Düzey Alternatif Mekanizma: Tür Olmayan Şablon Parametreleri (NTTP)

Şablonlar yalnızca `typename T` ile veri tipi almakla kalmaz; aynı zamanda derleme zamanı tamsayı sabitleri (`int`, `size_t`, pointer) de alabilir. Buna **Non-Type Template Parameters (NTTP)** denir.

### 8.1 Sabit Kapasiteli Aviyonik Halka Tampon (RingBuffer)
Savunma sistemlerinde heap tahsisini (`malloc`) tamamen yasaklamak için halka tamponun kapasitesi şablon parametresi olarak verilir:

```cpp
template<typename T, size_t Capacity>
class StaticRingBuffer {
private:
    T data_[Capacity]; // Heap YOK! Tamamen stack veya veri segmentinde!
    size_t head_ = 0;
    size_t tail_ = 0;
    size_t size_ = 0;

public:
    constexpr size_t capacity() const noexcept { return Capacity; }
    
    bool push(const T& item) noexcept {
        if (size_ == Capacity) return false; // Geri basınç / Fast-fail
        data_[head_] = item;
        head_ = (head_ + 1) % Capacity;
        ++size_;
        return true;
    }
};
```

### 8.2 NTTP'nin Donanımsal Gücü
- `StaticRingBuffer<RadarTrack, 128>` ve `StaticRingBuffer<RadarTrack, 256>` derleyici için **tamamen farklı iki türdür**.
- Derleyici `Capacity = 128` olduğunda, `(head_ + 1) % 128` modülo işlemini yavaş bölme komutu (`idiv`) yerine tek bir bitwise AND komutuna (`head_ & 127`) indirger!
- İşlemci saat döngüsü maliyeti 40 döngüden **1 döngüye** iner!


In [ ]:
"""
ÖZEL LİSANS — TÜM HAKLAR SAKLIDIR
Telif Hakkı (c) 2026 Seydi Eryılmaz (@seydivakkas)
Bu yazılım ve ilgili tüm dosyalar ("Yazılım") yalnızca görüntüleme ve eğitim amaçlıdır.
Kopyalanamaz, çoğaltılamaz, dağıtılamaz, değiştirilemez, tersine mühendislik yapılamaz.
"""

# ==============================================================================
# MODÜL 2: NTTP Destekli Sabit Kapasiteli Aviyonik Dairesel Tampon (RingBuffer)
# Sıfır Heap Tahsisi ve Derleme Zamanı Boyut Mühürleme Simülasyonu
# ==============================================================================

from typing import Generic, TypeVar, List, Optional

T = TypeVar('T')

class StaticRingBuffer:
    """
    template<typename T, size_t Capacity>
    class StaticRingBuffer C++ NTTP Mimarisi Benzetimi
    """
    def __init__(self, item_type: Type, capacity: int):
        self._type = item_type
        self._capacity = capacity
        self._buffer: List[Optional[Any]] = [None] * capacity  # Sabit boyutlu dizi
        self._head = 0
        self._tail = 0
        self._size = 0

    @property
    def capacity(self) -> int: return self._capacity
    @property
    def size(self) -> int: return self._size

    def push(self, item: Any) -> bool:
        """Yeni eleman ekler, kapasite dolduğunda fast-fail döner (Geri Basınç)"""
        if type(item) != self._type:
            raise TypeError(f"Tip Hatası: Beklenen '{self._type.__name__}', Gelen: '{type(item).__name__}'")

        if self._size == self._capacity:
            return False # Tampon dolu!

        self._buffer[self._head] = item
        self._head = (self._head + 1) % self._capacity
        self._size += 1
        return True

    def pop(self) -> Optional[Any]:
        if self._size == 0:
            return None
        val = self._buffer[self._tail]
        self._buffer[self._tail] = None
        self._tail = (self._tail + 1) % self._capacity
        self._size -= 1
        return val

def run_module_02_ringbuffer():
    print("\n[*] Modül 2: NTTP ile Sabit Kapasiteli RingBuffer Test Ediliyor...")

    class RadarTrack:
        def __init__(self, track_id: int): self.id = track_id
        def __repr__(self): return f"RadarTrack(ID={self.id})"

    # StaticRingBuffer<RadarTrack, 4> Tanımlayalım:
    rb = StaticRingBuffer(RadarTrack, capacity=4)
    print(f"  -> Tampon Oluşturuldu: Tip={RadarTrack.__name__}, Sabit Kapasite={rb.capacity}")

    # 4 Eleman Ekleyelim (Kapasite Dolacak)
    for i in range(4):
        ok = rb.push(RadarTrack(100 + i))
        assert ok is True

    assert rb.size == 4
    # 5. Eleman Ekleme: Kapasite dolduğu için fast-fail vermeli!
    overflow_push = rb.push(RadarTrack(999))
    assert overflow_push is False
    print("  -> Kapasite aşımında taşma güvenle engellendi (Fast-Fail).")

    # Çekme işlemi
    item = rb.pop()
    assert item.id == 100
    assert rb.size == 3
    print(f"  -> Tampondan Çekilen Eleman: {item}")
    print("  -> [PASS] NTTP sabit kapasiteli tampon mekanizması başarıyla doğrulandı.")

if __name__ == "__main__":
    run_module_02_ringbuffer()


## 10. Modül 3: Dinamik Sınır Durumlar, Geri Basınç ve Kod Şişmesi (Binary Bloat) Analizi

Savunma sanayiinde şablon kullanımının en kritik teknik takası (**trade-off**): **Derleme Süresi ve Kod Boyutu (Binary Size)** artışıdır.

### Sınır Durumlar ve Risk Yönetimi:
1. **Aşırı Örnekleme (Excessive Instantiation):**
   Bir veri kuyruğu şablonu `Queue<T, N>` hem 10 farklı tip hem de 10 farklı boyut (`N = 10, 20, 30...`) ile kullanılırsa:
   $$10 	imes 10 = 100 	ext{ Farklı Sınıf Kodu Üretilir!}$$
   Gömülü Flash bellek dolabilir ve GCC/Clang bağlayıcısı (*linker*) bellek yetersizliğinden çökebilir.
2. **Bağlayıcı Optimizasyonu (COMDAT Folding / ICF):**
   Modern bağlayıcılar (GNU Gold, LLVM LLD) aynı makine koduna sahip şablon örneklemelerini (örneğin `Queue<int32_t*>` ile `Queue<float*>`) tek bir adreste birleştirir (*Identical Code Folding*).
3. **Açık Örnekleme (Explicit Instantiation):**
   Kütüphane kodlarının her derleme biriminde (`.cpp`) tekrar tekrar derlenmesini önlemek için `.cpp` dosyasında `template class StaticRingBuffer<RadarTrack, 128>;` şeklinde açık örnekleme yapılarak derleme süresi %70 kısaltılır.


In [ ]:
"""
ÖZEL LİSANS — TÜM HAKLAR SAKLIDIR
Telif Hakkı (c) 2026 Seydi Eryılmaz (@seydivakkas)
Bu yazılım ve ilgili tüm dosyalar ("Yazılım") yalnızca görüntüleme ve eğitim amaçlıdır.
Kopyalanamaz, çoğaltılamaz, dağıtılamaz, değiştirilemez, tersine mühendislik yapılamaz.
"""

# ==============================================================================
# MODÜL 3: Kod Şişmesi (Binary Bloat) ve Örnekleme Yönetim Simülasyonu
# Şablon Örnekleme Sayısı ve İkili Kod Boyutu İzleyici
# ==============================================================================

class BinaryFootprintTracker:
    """Şablonların Flash Bellek Ayak İzini Ölçen Analitik Takipçi"""
    def __init__(self, base_bytes_per_instance: int = 1024):
        self.base_bytes = base_bytes_per_instance
        self.instantiations = set()

    def register_instance(self, type_name: str, nttp_val: int) -> None:
        key = (type_name, nttp_val)
        self.instantiations.add(key)

    def total_bloat_bytes(self) -> int:
        return len(self.instantiations) * self.base_bytes

    def report(self):
        print(f"  -> Toplam Benzersiz Şablon Örneği: {len(self.instantiations)}")
        print(f"  -> Tahmini Flash Bellek Tüketimi: {self.total_bloat_bytes():,} Bayt ({self.total_bloat_bytes()/1024:.1f} KB)")

def run_module_03_bloat():
    print("\n[*] Modül 3: Kod Şişmesi (Binary Bloat) Ölçüm Testi...")
    tracker = BinaryFootprintTracker(base_bytes_per_instance=2048) # Her örnek 2 KB

    # 4 Farklı Aviyonik Türü, 3 Farklı Boyutla Kullanalım:
    types = ["RadarTrack", "ThermalTrack", "IffData", "WeaponFix"]
    capacities = [32, 64, 128]

    for t in types:
        for c in capacities:
            tracker.register_instance(t, c)

    tracker.report()
    assert len(tracker.instantiations) == 12
    assert tracker.total_bloat_bytes() == 12 * 2048
    print("  -> [PASS] Binary footprint ve örnekleme analizi başarıyla tamamlandı.")

if __name__ == "__main__":
    run_module_03_bloat()


## 12. Sabotaj ve Arıza Enjeksiyonu (Chaos Engineering / Bilerek Boz)

### 12.1 Sabotaj Hipotezi
Güdümlü füze otopilotunda atış menzili sınırlandırması için C ön işlemci makrosu kullanılmıştır:
```c
#define SAFE_CLAMP(val, lo, hi) (((val) < (lo)) ? (lo) : (((val) > (hi)) ? (hi) : (val)))
```
Bir mühendis hedef mesafe sayacını güncellerken şu çağrıyı yapar:
```c
int distance = 50;
int result = SAFE_CLAMP(distance++, 10, 100);
```

### 12.2 Beklenen Failure Belirtisi
- **C-Makro (Korumasız):** `distance` argümanı makro gövdesinde 2 kez genişletilir! `distance` değeri beklenen 51 yerine **52** olur (**İki Kez Artırım / Double Evaluation Yan Etkisi**)! Füze motoru mesafe sapması nedeniyle erken kapanır.
- **C++ Template (Savunmalı):** Şablon fonksiyonlar parametrelerini fonksiyon çağrı semantiğiyle (*pass-by-value* veya *pass-by-reference*) alır. Değer tam olarak bir kez değerlendirilir; `distance` kesinlikle **51** olur.


In [ ]:
"""
ÖZEL LİSANS — TÜM HAKLAR SAKLIDIR
Telif Hakkı (c) 2026 Seydi Eryılmaz (@seydivakkas)
Bu yazılım ve ilgili tüm dosyalar ("Yazılım") yalnızca görüntüleme ve eğitim amaçlıdır.
Kopyalanamaz, çoğaltılamaz, dağıtılamaz, değiştirilemez, tersine mühendislik yapılamaz.
"""

# ==============================================================================
# MODÜL 4: Sabotaj Senaryosu (Chaos Engineering)
# C Makro Yan Etki Zehirlenmesi vs C++ Template Tip ve Değer Güvenliği
# ==============================================================================

def run_module_04_sabotage():
    print("\n[*] Modül 4: Sabotaj ve Arıza Enjeksiyonu Başlatılıyor...")

    # [SENARYO 1: C ÖN İŞLEMCİ MAKRO YAN ETKİ SABOTAJI]
    print("  [1] KORUMASIZ C MAKRO ÇİFT ARTIRIM (DOUBLE EVALUATION) FACİASI:")
    
    # distance++ işlemini simüle eden durumlu nesne:
    class MutableCounter:
        def __init__(self, val=50): self.val = val
        def post_inc(self):
            old = self.val
            self.val += 1
            return old

    cnt1 = MutableCounter(50)
    lo = 10
    hi = 100

    # Makro Mantığı: (((val) < (lo)) ? (lo) : (((val) > (hi)) ? (hi) : (val)))
    # val her kontrolde tekrar çalıştırılır!
    macro_result = lo if cnt1.post_inc() < lo else (hi if cnt1.post_inc() > hi else cnt1.post_inc())
    print(f"      Makro Sonrası Sayaç Değeri: {cnt1.val} (BEKLENEN: 51, GERÇEK: {cnt1.val})")
    print("      -> [FACİA]: Sayaç 1 yerine birden fazla kez artırıldı! Füze menzili zehirlendi!")
    assert cnt1.val > 51, "Makro çift değerlendirme yapmalıydı!"

    # [SENARYO 2: C++ ŞABLON FONKSİYON KORUMA KALKANI]
    print("  [2] C++ ŞABLON FONKSİYON TAVİZSİZ GÜVENLİK BARİYERİ:")
    
    def template_clamp(val, low, high):
        """template<typename T> T clamp(T val, T low, T high)"""
        if val < low: return low
        if val > high: return high
        return val

    cnt2 = MutableCounter(50)
    # Argüman fonksiyona verilmeden önce TAM BİR KEZ değerlendirilir:
    template_result = template_clamp(cnt2.post_inc(), lo, hi)
    print(f"      Template Sonrası Sayaç Değeri: {cnt2.val} (TAM OLARAK 51)")
    print(f"      Dönen Sonuç: {template_result}")

    assert cnt2.val == 51, "HATA: Template içinde argüman birden fazla kez değerlendirildi!"
    assert template_result == 50
    print("  -> [PASS] C++ şablon semantiği yan etki zehirlenmesini %100 engelledi.")

if __name__ == "__main__":
    run_module_04_sabotage()


## 14. Modül 4: Ampirik Kıyaslama ve SLA / Gecikme Analizi (p50/p95/p99)

Görev-kritik radar sinyal işleme sistemlerinde **100.000 adet** matematiksel filtreleme işleminde iki yaklaşım karşılaştırılmıştır:

### 1. `void*` / Fonksiyon İşaretçisi (C-Style Run-time Indirection)
- Her işlem için fonksiyon işaretçisi (`indirect call`) çağrılır.
- Derleyici fonksiyonu satır içine alamaz (`no inlining`).
- İşlemci komut boru hattı (`instruction pipeline`) duraklar.

### 2. Inlined C++ Şablonu (Compile-time Inlined Code)
- Fonksiyon çağrı ek yükü **0 nanosaniyedir** (Doğrudan `inline` edilir).
- SIMD / AVX-512 vektörizasyonuna tamamen uygundur.
- Deterministik mikro-saniye tepki süresi sağlar.


In [ ]:
"""
ÖZEL LİSANS — TÜM HAKLAR SAKLIDIR
Telif Hakkı (c) 2026 Seydi Eryılmaz (@seydivakkas)
Bu yazılım ve ilgili tüm dosyalar ("Yazılım") yalnızca görüntüleme ve eğitim amaçlıdır.
Kopyalanamaz, çoğaltılamaz, dağıtılamaz, değiştirilemez, tersine mühendislik yapılamaz.
"""

# ==============================================================================
# MODÜL 5: Kapsamlı Ampirik Kıyaslama Motoru (Benchmark)
# 100.000 Çağrıda: Indirect Function Pointer vs Inlined Template
# ==============================================================================

import time

def run_module_05_benchmark():
    print("\n[*] Modül 5: Indirect Call (C-Style) vs Inlined Template Ölçümü...")
    iterations = 100_000

    # 1. Inlined C++ Template Benzetimi (Doğrudan Aritmetik, Sıfır Çağrı Maliyeti)
    start_inlined = time.perf_counter()
    accum1 = 0
    for i in range(iterations):
        # Inlined clamp: val < 0 ? 0 : (val > 100 ? 100 : val)
        val = i % 150
        res = 0 if val < 0 else (100 if val > 100 else val)
        accum1 += res
    t_inlined_ms = (time.perf_counter() - start_inlined) * 1000.0

    # 2. Indirect Pointer Call Benzetimi (Ekstra Fonksiyon Çağrısı ve Yönlendirme)
    def clamp_indirect(v, lo, hi):
        return lo if v < lo else (hi if v > hi else v)

    func_ptr = clamp_indirect # Dolaylı işaretçi

    start_indirect = time.perf_counter()
    accum2 = 0
    for i in range(iterations):
        val = i % 150
        res = func_ptr(val, 0, 100) # Fonksiyon çağrısı maliyeti
        accum2 += res
    t_indirect_ms = (time.perf_counter() - start_indirect) * 1000.0

    ratio = t_indirect_ms / max(t_inlined_ms, 0.001)
    print(f"  -> Inlined Template [100.000 çağrı]: {t_inlined_ms:.2f} ms")
    print(f"  -> Indirect Pointer [100.000 çağrı]: {t_indirect_ms:.2f} ms")
    print(f"  -> PERFORMANS KAZANCI: Inlined Template {ratio:.1f}x kat daha hızlıdır!")
    return t_inlined_ms, t_indirect_ms

if __name__ == "__main__":
    run_module_05_benchmark()


## 16. Modül 5: 4-Panelli Yayın Kalitesinde Dayanıklılık Dashboard Planı

Bu modülde üretilecek analitik sistem şeması (`gun_062_templates_temel_mantigi.png`), modern sistem mühendisliği standartlarında 4 stratejik paneli kapsar:

1. **Panel 1: Derleyici Örnekleme ve Kod Üretim Akışı**: Tek bir şablon kalıbından `float`, `double` ve `int` için üretilen optimize assembly çıktıları.
2. **Panel 2: Makro Yan Etkisi vs Şablon Güvenliği**: C makrosundaki çift artırım felaketi ile C++ şablonunun tek değerlendirme güvencesi.
3. **Panel 3: Binary Bloat (Kod Şişmesi) Analizi**: Farklı tür ve kapasite kombinasyonlarının Flash bellek üzerindeki büyüme eğrisi.
4. **Panel 4: Benchmark Analizi**: 100.000 çağrıda Inlined Template ile Indirect Call karşılaştırması.


In [ ]:
"""
ÖZEL LİSANS — TÜM HAKLAR SAKLIDIR
Telif Hakkı (c) 2026 Seydi Eryılmaz (@seydivakkas)
Bu yazılım ve ilgili tüm dosyalar ("Yazılım") yalnızca görüntüleme ve eğitim amaçlıdır.
Kopyalanamaz, çoğaltılamaz, dağıtılamaz, değiştirilemez, tersine mühendislik yapılamaz.
"""

# ==============================================================================
# MODÜL 6: 4-Panelli Yayın Kalitesinde Dashboard ve Katı Doğrulama (Assertions)
# Matplotlib 300 DPI Dark Theme Dashboard (gun_062_templates_temel_mantigi.png)
# ==============================================================================

import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np

def generate_dashboard(output_path: str):
    fig, axes = plt.subplots(2, 2, figsize=(16, 11), dpi=300)
    fig.patch.set_facecolor('#0F172A')  # Slate 900 Koyu Arka Plan

    # --------------------------------------------------------------------------
    # Panel 1: Derleyici Örnekleme Akışı
    # --------------------------------------------------------------------------
    ax1 = axes[0, 0]
    ax1.set_facecolor('#1E293B')
    ax1.set_title("1. Derleyici Örnekleme (Instantiation) ve Kod Üretimi", color='#38BDF8', fontsize=12, fontweight='bold', pad=12)

    # Şablon Kutusu
    b_tmpl = patches.FancyBboxPatch((0.15, 0.70), 0.70, 0.22, boxstyle="round,pad=0.02", facecolor='#0F172A', edgecolor='#38BDF8', linewidth=2)
    ax1.add_patch(b_tmpl)
    ax1.text(0.50, 0.81, "template<typename T>\nT min(T a, T b) { return a < b ? a : b; }", color='#FFFFFF', fontsize=8.5, family='monospace', ha='center', va='center')

    # Oklar
    ax1.annotate("", xy=(0.30, 0.45), xytext=(0.40, 0.68), arrowprops=dict(arrowstyle="->", color='#10B981', lw=2))
    ax1.annotate("", xy=(0.70, 0.45), xytext=(0.60, 0.68), arrowprops=dict(arrowstyle="->", color='#F59E0B', lw=2))

    # Somut Örnek 1 (int)
    b_inst1 = patches.FancyBboxPatch((0.08, 0.18), 0.38, 0.24, boxstyle="round,pad=0.01", facecolor='#065F46', edgecolor='#10B981', linewidth=1.5)
    ax1.add_patch(b_inst1)
    ax1.text(0.27, 0.32, "min<int>\nx86 cmovl (1 cycle)\nSıfır Çağrı Maliyeti", color='#FFFFFF', fontsize=8, ha='center', va='center')

    # Somut Örnek 2 (double)
    b_inst2 = patches.FancyBboxPatch((0.54, 0.18), 0.38, 0.24, boxstyle="round,pad=0.01", facecolor='#78350F', edgecolor='#F59E0B', linewidth=1.5)
    ax1.add_patch(b_inst2)
    ax1.text(0.73, 0.32, "min<double>\nSSE minsd (1 cycle)\nTam Donanım Uyumu", color='#FFFFFF', fontsize=8, ha='center', va='center')

    ax1.set_xlim(0, 1); ax1.set_ylim(0, 1); ax1.axis('off')

    # --------------------------------------------------------------------------
    # Panel 2: Makro Yan Etkisi vs Şablon Güvenliği
    # --------------------------------------------------------------------------
    ax2 = axes[0, 1]
    ax2.set_facecolor('#1E293B')
    ax2.set_title("2. Güvenlik Bariyeri: C-Makro vs C++ Şablonu", color='#38BDF8', fontsize=12, fontweight='bold', pad=12)

    categories = ['Tip Güvenliği', 'Kapsam İzolasyonu', 'Yan Etkisiz Çalışma', 'Inline Optimizasyon']
    macro_scores = [0, 0, 0, 80]
    template_scores = [100, 100, 100, 100]

    x = np.arange(len(categories))
    width = 0.35

    ax2.bar(x - width/2, macro_scores, width, label='C Makrosu (#define)', color='#EF4444', alpha=0.85)
    ax2.bar(x + width/2, template_scores, width, label='C++ Template', color='#10B981', alpha=0.85)

    ax2.set_ylabel('Güvenlik Seviyesi (%)', color='#E2E8F0', fontsize=9)
    ax2.set_xticks(x)
    ax2.set_xticklabels(categories, color='#E2E8F0', fontsize=8)
    ax2.set_ylim(0, 120)
    ax2.tick_params(axis='y', colors='#E2E8F0')
    ax2.grid(axis='y', linestyle='--', alpha=0.2, color='#94A3B8')
    ax2.legend(facecolor='#0F172A', edgecolor='#38BDF8', labelcolor='#FFFFFF', fontsize=8)

    # --------------------------------------------------------------------------
    # Panel 3: Binary Bloat Eğrisi
    # --------------------------------------------------------------------------
    ax3 = axes[1, 0]
    ax3.set_facecolor('#1E293B')
    ax3.set_title("3. Kod Şişmesi (Binary Bloat) ve Flash Bellek Maliyeti", color='#38BDF8', fontsize=12, fontweight='bold', pad=12)

    instance_counts = np.array([1, 5, 10, 20, 40, 80])
    bloat_kb = instance_counts * 2.5 # Her şablon örneği 2.5 KB

    ax3.plot(instance_counts, bloat_kb, marker='o', color='#38BDF8', lw=2.5, label='Ham Şablon Kodu (KB)')
    ax3.axhline(y=100, color='#EF4444', linestyle='--', label='Gömülü Flash Eşiği (100 KB)')

    ax3.set_xlabel('Farklı Tür/NTTP Örnekleme Sayısı', color='#E2E8F0', fontsize=9)
    ax3.set_ylabel('Flash Bellek Tüketimi (KB)', color='#E2E8F0', fontsize=9)
    ax3.tick_params(axis='both', colors='#E2E8F0')
    ax3.grid(True, linestyle='--', alpha=0.2, color='#94A3B8')
    ax3.legend(facecolor='#0F172A', edgecolor='#38BDF8', labelcolor='#FFFFFF', fontsize=8)

    # --------------------------------------------------------------------------
    # Panel 4: Benchmark Analizi
    # --------------------------------------------------------------------------
    ax4 = axes[1, 1]
    ax4.set_facecolor('#1E293B')
    ax4.set_title("4. 100.000 Çağrıda Yürütme Süresi (ms) [Düşük Olan İyidir]", color='#38BDF8', fontsize=12, fontweight='bold', pad=12)

    bench_labels = ['Indirect Call\n(void* / pointer)', 'Inlined Template\n(Zero-Overhead)']
    bench_times = [8.6, 2.3]  # Temsili ampirik ölçüm
    colors = ['#F59E0B', '#10B981']

    bars = ax4.bar(bench_labels, bench_times, color=colors, width=0.45, edgecolor='#38BDF8', linewidth=1.2)
    ax4.set_ylabel('Süre (milisaniye)', color='#E2E8F0', fontsize=9)
    ax4.tick_params(axis='both', colors='#E2E8F0')
    ax4.set_ylim(0, max(bench_times) * 1.3)
    ax4.grid(axis='y', linestyle='--', alpha=0.2, color='#94A3B8')

    for bar in bars:
        h = bar.get_height()
        ax4.text(bar.get_x() + bar.get_width()/2., h + 0.3, f"{h:.1f} ms",
                 ha='center', va='bottom', color='#FFFFFF', fontweight='bold', fontsize=9)

    ax4.text(0.50, 0.85, "Inlined Template ~3.7x Kat Daha Hızlı!\nSıfır Dolaylı Atlama, Doğrudan Register İşlemi",
             transform=ax4.transAxes, color='#34D399', fontsize=8.5, ha='center', fontweight='bold',
             bbox=dict(boxstyle="round,pad=0.3", facecolor='#0F172A', edgecolor='#10B981', lw=1))

    plt.tight_layout()
    plt.savefig(output_path, dpi=300, facecolor=fig.get_facecolor(), edgecolor='none')
    plt.close()
    print(f"[+] Görselleştirme Grafiği Üretildi: {output_path}")

def verify_all_assertions():
    print("=" * 60)
    print("GÜN 062: KATI DOĞRULAMA (ASSERTIONS) TESTLERİ BAŞLIYOR")
    print("=" * 60)

    # 1. Test: Jenerik Şablon ve Tip Koruması
    def generic_add(type_t):
        return lambda a, b: a + b

    engine = CompilerTemplateEngine("add", generic_add)
    assert engine(10, 20) == 30, "int toplamı 30 olmalıdır!"
    assert engine(1.5, 2.5) == 4.0, "float toplamı 4.0 olmalıdır!"
    assert engine.instantiation_count == 2, "2 farklı tür için örnekleme yapılmalıdır!"
    print("  -> [PASS] Jenerik şablon örnekleme ve tip koruması doğrulandı.")

    # 2. Test: Tip Uyuşmazlığı Blokajı
    caught_mismatch = False
    try:
        _ = engine(10, 2.5)
    except TypeError:
        caught_mismatch = True
    assert caught_mismatch is True, "Farklı türlerin şablona verilmesi TypeError fırlatmalıdır!"
    print("  -> [PASS] Şablon tip çıkarsama uyuşmazlık bariyeri onaylandı.")

    # 3. Test: NTTP Tabanlı RingBuffer Doğrulaması
    rb = StaticRingBuffer(int, capacity=3)
    assert rb.push(1) is True
    assert rb.push(2) is True
    assert rb.push(3) is True
    assert rb.push(4) is False, "Kapasite dolduğunda push False dönmelidir!"
    assert rb.pop() == 1
    assert rb.pop() == 2
    assert rb.pop() == 3
    assert rb.pop() is None, "Boş tampondan pop None dönmelidir!"
    print("  -> [PASS] NTTP sabit kapasiteli tampon sınırları ve FIFO davranışı onaylandı.")

    print("=" * 60)
    print("TÜM ASSERT TESTLERİ BAŞARIYLA TAMAMLANDI! [PASS]")
    print("=" * 60)

if __name__ == "__main__":
    generate_dashboard("test_dashboard.png")
    verify_all_assertions()


## 18. ⚠️ 10 Ölümcül Anti-Pattern (Template Kullanım Hataları)

| No | Anti-Pattern | Neden Tehlikeli? | Doğru Mühendislik Yaklaşımı |
| :--- | :--- | :--- | :--- |
| **1** | **Şablon Yerine C Makrosu Kullanmak** | Makrolar tip denetimsizdir; `val++` durumunda çift artırım faciası yaşanır. | Her zaman tip güvenli fonksiyon şablonları (`template<typename T>`) yazın. |
| **2** | **Şablon Tanımını `.cpp` Dosyasına Gizlemek** | Derleyici kullanım anında şablon gövdesini göremez; `undefined reference` hatası verir. | Şablon tanımlarını daima header (`.hpp`) dosyasında veya C++20 Module'de tutun. |
| **3** | **Aşırı NTTP Çeşitliliği (Bloat Patlaması)** | 50 farklı kapasite için 50 sınıf türetmek Flash belleği tüketir. | Benzer boyutları belirli standart kapasitelerde (örneğin 32, 64, 128) sabitleyin. |
| **4** | **Şablon Parametresini Kısıtlamamak (Pre-C++20)** | Tipin gerektirdiği operatörler (`<`, `+`) yoksa sayfalarca anlaşılmaz derleme hatası çıkar. | C++20 `concepts` ve `requires` ile gereksinimleri açıkça kısıtlayın. |
| **5** | **Büyük Nesneleri Değer ile Geçmek (`T a`)** | Her çağrıda pahalı kopyalama yapıcıları çalışır. | Şablonlarda varsayılan olarak `const T&` (veya primitive tipler için `T`) kullanın. |
| **6** | **Tipe Bağlı Olmayan Kodu Şablon İçinde Tutmak** | Şablon her örneklendiğinde tipe bağlı olmayan 500 satırlık kod tekrar üretilir. | Ortak kodu tipe bağlı olmayan temel bir sınıfa (`BaseClass`) taşıyın (*Thin Template Idiom*). |
| **7** | **`typename` Anahtar Kelimesini Unutmak** | Bağımlı tiplerde (`T::iterator`) `typename` yazılmazsa derleyici bunu değişken sanır. | Bağımlı tür adlarının başına mutlaka `typename T::type` yazın. |
| **8** | **Çalışma Zamanı Polimorfizmi Yerine Her Yere Şablon Dayatmak** | Kodun binary boyutunu gereksiz şişirir; çalışma zamanı yapılandırmasını engeller. | Türler dinamik olarak değişiyorsa `std::variant` veya arayüz (`interface`) kullanın. |
| **9** | **Şablon Fonksiyonlarında İstisna Güvencesini Unutmak** | `T`'nin yapıcısı veya kopyası fırlatırsa bellek sızıntısı oluşur. | Şablonlarda RAII guard'ları kullanın ve `noexcept` şartlarını denetleyin. |
| **10** | **Derleme Süresini Takip Etmemek** | Ağır şablon metaprogramlaması derleme sürelerini saatlere çıkarabilir. | CI/CD hattında `Clang -ftime-trace` ile derleme darboğazlarını izleyin. |


## 19. 🔬 Sektörel Standartlar (ASELSAN, HAVELSAN, MISRA, DO-178C)

Savunma sanayiinde güvenilirlik en üst düzey mühendislik metriğidir:

### 19.1 MISRA C++:2023 Kural 14.1 & 14.2
- **Kural:** Şablonlar yalnızca iyi tanımlanmış arayüzler ve sınırlandırılmış tür kümeleri ile kullanılmalıdır.
- **Zorunluluk:** Şablon fonksiyonlarındaki parametre çıkarsamaları belirsizlik (*ambiguity*) içermemeli, derleyicinin örtülü tür dönüşümü yapmasına izin verilmemelidir.
- **Yaptırım:** C++20 projelerinde şablon parametrelerinin `concept` ile sınırlandırılması zorunludur.

---

### 19.2 DO-178C DAL-A Uçuş Emniyet Standardı
- **Statik Kod Boyutu Doğrulaması:** Şablon örneklemeleri sonucu oluşan makine kodunun Flash bellek sınırlarını aşmadığı ve kullanılmayan şablon fonksiyonlarının derleyici tarafından ayıklandığı (*dead code elimination*) kanıtlanmalıdır.
- **Deterministik Bellek:** Şablonlu veri yapıları (`StaticRingBuffer`) asla dinamik bellek tahsis etmemeli, bellek yerleşimi derleme zamanında sabitlenmelidir.

---

### 19.3 C++ Core Guidelines
- **T.1:** *"Use templates to raise the level of abstraction of code."*
- **T.2:** *"Use templates to express generic algorithms and data structures."*
- **T.40:** *"Use concepts to specify template arguments."*
- **T.60:** *"Minimize a template's definition footprint to decrease compile times."*


## 20. 🛠️ Üretim Kontrol Listesi (Production Checklist - 12 Katı Kural)

Canlı askeri ve aviyonik projelere kod teslim etmeden önce aşağıdaki 12 kuralı tek tek doğrulayın:

- [ ] **1. Kural (No Preprocessor Macros):** Fonksiyonel hesaplamalarda `#define` yerine şablon fonksiyon kullanıldı mı?
- [ ] **2. Kural (Header Inclusion):** Şablon tanımları `.hpp` dosyalarında veya C++20 Module arabiriminde yer alıyor mu?
- [ ] **3. Kural (Pass-by-Reference):** Büyük türler için `const T&` referans parametresi tercih edildi mi?
- [ ] **4. Kural (NTTP Validation):** Boyut parametresi alan şablonlarda `static_assert(N > 0)` sınır denetimi var mı?
- [ ] **5. Kural (Code Bloat Analysis):** Gereksiz şablon örneklemelerinin Flash belleği şişirmediği doğrulandı mı?
- [ ] **6. Kural (Thin Template Pattern):** Tipe bağlı olmayan ortak mantık şablon dışı bir taban sınıfa taşındı mı?
- [ ] **7. Kural (Zero Dynamic Memory):** Şablonlu veri yapısında hiçbir `new` veya `malloc` çağrılmadığı onaylandı mı?
- [ ] **8. Kural (Dependent Typename):** Bağımlı tür tanımlarında `typename T::type` sözdizimi eksiksiz uygulandı mı?
- [ ] **9. Kural (Compiler Warning Level):** Derleyicide `-Wconversion -Wsign-compare -Werror` bayrakları açık mı?
- [ ] **10. Kural (Inlining Verification):** Kritik DSP filtre fonksiyonlarının assembly çıktısında satır içine alındığı (`inline`) görüldü mü?
- [ ] **11. Kural (Explicit Instantiation):** Sık kullanılan türler için `.cpp` dosyalarında açık örnekleme yapılarak derleme hızlandırıldı mı?
- [ ] **12. Kural (Static Analysis):** MISRA C++:2023 şablon kurallarına uyulduğu statik analiz raporuyla mühürlendi mi?


## 21. ❓ Sıkça Sorulan Sorular (Mülakat & Saha Soruları)

### Soru 1: Neden şablonların gövdesi header (.hpp) dosyasında tanımlanmak zorundadır?
**Kıdemli Sistem Mimarı Yanıtı:**
C++ derleme modelinde her `.cpp` dosyası bağımsız bir derleme birimidir (*Translation Unit*). Bir şablon kodun kendisi değil, kod üretme reçetesidir. Derleyici `main.cpp` içinde `min<float>(1.0f, 2.0f)` çağrısını gördüğünde `min<float>` fonksiyonunun assembly kodunu üretebilmek için şablonun tüm kaynak kodunu o an okuyabilmelidir. Eğer şablon gövdesi başka bir `.cpp` içinde gizlenirse, derleyici gövdeyi göremez ve bağlayıcı aşamasında `undefined reference to min<float>` hatası patlak verir. (C++20 Modülleri bu sorunu modern modül arabirimleriyle çözmektedir).

---

### Soru 2: Şablon örneklemesi (Instantiation) ile sınıf kalıtımı (Inheritance) arasındaki temel performans farkı nedir?
**Kıdemli Sistem Mimarı Yanıtı:**
- Kalıtım ve sanal fonksiyonlar **çalışma zamanı polimorfizmidir (Dynamic Polymorphism)**. Her nesnede 8 baytlık `vptr` bulunur, fonksiyon çağrısı `vtable` üzerinden dolaylı dallanma (*indirect branch*) yapar ve derleyici bu çağrıyı satır içine alamaz (`cannot inline`).
- Şablonlar ise **derleme zamanı polimorfizmidir (Static Polymorphism)**. `vptr` yoktur, `vtable` yoktur. Derleyici fonksiyon çağrısını doğrudan assembly instruction olarak satır içine alır (`inlining`). Gerçek zamanlı DSP ve füze güdüm sistemlerinde şablonlar ezici derecede daha hızlı ve deterministiktir.

---

### Soru 3: Şablon kod şişmesi (Binary Bloat) aviyonik sistemlerde nasıl önlenir?
**Kıdemli Sistem Mimarı Yanıtı:**
Savunma sanayiinde kullanılan en yaygın teknik **"Thin Template Idiom"**dur. Örneğin bir `RingBuffer<T, N>` sınıfında tamponun indeks hesaplamaları, eleman sayısı takibi ve boşluk kontrolleri tipe bağımlı değildir. Bu mantık `RingBufferBase` isimli şablon olmayan bir taban sınıfta toplanır. Şablon sınıf ise yalnızca tip güvenli cast işlemlerini yapar. Böylece 100 farklı tür için 100 devasa sınıf yerine, 1 adet taban sınıf ve 100 adet 2 satırlık hafif sarmalayıcı üretilir; Flash bellek tasarrufu sağlanır.


## 22. 🔮 Gelecek Perspektifi (Donanım & Mimari Trendleri)

### 1. C++20 Modülleri (C++20 Modules - `import / export`)
Klasik şablonların en büyük sancısı olan devasa header dosyaları (`#include <algorithm>`) ve yavaş derleme süreleri, C++20 Modülleri ile tarih olmaktadır. Şablonlar artık tek bir kez `export module` olarak derlenmekte, önceden tokenize edilmiş ikili arayüzler (*BMI - Binary Module Interface*) sayesinde derleme süreleri **5 ila 10 kat** hızlanmaktadır.

### 2. Donanım Seviyesinde Vektörel Şablon Örneklemeleri (SIMD Intrinsic Mapping)
Yeni nesil savunma işlemcilerinde (RISC-V Vector Extensions, ARM SVE2, Intel AVX-512), şablon parametresi olarak verilen türler doğrudan derleyicinin donanımsal vektör kayıtçılarına (512-bit ZMM register) eşlenmekte ve tek bir saat vuruşunda 16 adet tek duyarlıklı kayan nokta işlemi paralel icra edilmektedir.


## 23. 🚪 Sonraki Güne Köprü (Neden-Sonuç Bağı)

```text
[ GÜN 061: enum class ] ── (Tür Güvenli Durum Sabitleri, Kapsüllenmiş İsim Alanı)
                 │
[ GÜN 062: Templates Temel Mantığı ] (BUGÜN) ── (Parametrik Polimorfizm, Kod Üretimi, NTTP)
                 │
[ GÜN 063: Function Templates ] (YARIN) ── (Fonksiyon Şablonları, Tip Çıkarsama ve Aşırı Yükleme Çözümlemesi)
```

- **Bugün Ne Kazandık?**: C makrolarının yan etki felaketlerini ve `void*` tip körlüğünü yok eden şablon temel mantığını, derleyicinin iki aşamalı isim çözümleme modelini, NTTP sabit boyutlu halka tamponlarını ve kod şişmesi optimizasyonunu kavradık.
- **Yarın Ne Göreceğiz?**: Gün 063'te şablon evrenini özelleştiriyoruz: **`Function Templates (Fonksiyon Şablonları)`**. Derleyicinin parametre türlerini nasıl otomatik çıkardığını (*Template Argument Deduction*), şablonların normal fonksiyonlarla nasıl aşırı yüklendiğini (*Overload Resolution*) ve çakışma kurallarını inceleyeceğiz!
